# RAG Chatbot for Car Technical Documentation

## Problem Statement

Car manuals are dense, technical documents that drivers rarely consult — yet understanding warning messages quickly can prevent accidents or costly damage. This notebook builds a **Retrieval-Augmented Generation (RAG) chatbot** that lets users ask natural-language questions about their car's warning messages and receive accurate, context-aware answers.

The corpus used is the **MG ZS warning messages** section of the official car manual (`data/mg-zs-warning-messages.html`). The system is able to answer queries such as:
> *"The Gasoline Particulate Filter Full warning has appeared. What does this mean and what should I do about it?"*

---

## Architecture Overview

The pipeline implements **RAG Fusion** — an enhanced retrieval strategy that improves recall by:

1. **Multi-query generation** — Given the user's question, the LLM generates 4 semantically diverse search queries.
2. **Parallel retrieval** — Each query is run independently against a Chroma vector store.
3. **Reciprocal Rank Fusion (RRF)** — Results from all queries are merged and re-ranked by a fusion score.
4. **Answer generation** — The top-ranked documents are injected as context into a final prompt answered by the LLM.

**Stack:** LangChain · Google Gemini (`gemini-2.5-flash`) · Gemini Embeddings · ChromaDB

---

## Table of Contents
1. [Installation](#1-installation)
2. [Imports & Configuration](#2-imports--configuration)
3. [Document Loading & Splitting](#3-document-loading--splitting)
4. [Vector Store & Retriever](#4-vector-store--retriever)
5. [Multi-Query Generation](#5-multi-query-generation)
6. [Reciprocal Rank Fusion](#6-reciprocal-rank-fusion)
7. [Final RAG Chain & Inference](#7-final-rag-chain--inference)

## 1. Installation

Install all required packages. Run this cell once when setting up a new environment.

In [ ]:
# Install required packages
# langchain_community   : community integrations (HTML loader, etc.)
# tiktoken              : tokenizer used by the text splitter
# langchain-openai      : OpenAI integration (used indirectly by some splitters)
# langchainhub          : access to community prompt templates
# chromadb              : local vector store
# langchain             : core framework
# langchain_core        : low-level primitives (prompts, runnables, parsers)
# langchain-google-genai: Google Gemini integration
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain
! pip install langchain_core langchain-google-genai itemgetter

## 2. Imports & Configuration

Load all dependencies and configure the LLM and embedding model.

> ⚠️ **Security note:** API keys must never be hard-coded in production code. Use a `.env` file loaded with `python-dotenv`, or environment variables set in your shell/CI system. The keys below are placeholders — replace them with your own.

In [ ]:
# --- Core LangChain primitives ---
from langchain_core.prompts import ChatPromptTemplate        # Template-based prompt construction
from langchain_core.runnables import RunnablePassthrough     # Pass input unchanged through a chain
from langchain_core.output_parsers import StrOutputParser    # Extract plain string from LLM response
from langchain_core.load import dumps, loads                 # Serialize/deserialize LangChain objects

# --- Google Gemini integration ---
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# --- Document handling ---
from langchain_community.document_loaders import UnstructuredHTMLLoader  # Parse raw HTML into Documents
from langchain_text_splitters import RecursiveCharacterTextSplitter       # Chunk documents intelligently

# --- Vector store ---
from langchain_chroma import Chroma  # In-memory / persistent vector store
from operator import itemgetter  # Functional key extractor for chain composition
import os

# ── LangSmith tracing (optional but recommended for debugging) ──
# Set LANGCHAIN_TRACING_V2=true to enable run traces on smith.langchain.com
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = 'YOUR_LANGSMITH_API_KEY'  
os.environ['GOOGLE_API_KEY']    = 'YOUR_GOOGLE_API_KEY'

# ── LLM: Google Gemini 2.5 Flash ──
# temperature=0 for deterministic, factual answers
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)

# ── Embeddings: Gemini Embedding 001 ──
# Used to encode document chunks and queries into a shared semantic space
embd = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    google_api_key=os.environ['GOOGLE_API_KEY']
)

## 3. Document Loading & Splitting

The HTML car manual is loaded, then split into overlapping chunks. Overlap ensures that warning messages split across chunk boundaries are still retrievable in their entirety.

In [ ]:
# Load the HTML manual using LangChain's UnstructuredHTMLLoader.
# This strips HTML tags and returns a list of Document objects with plain text content.
loader = UnstructuredHTMLLoader(file_path='data/mg-zs-warning-messages.html')
car_docs = loader.load()

In [ ]:
# Split the loaded documents into chunks for embedding.
#
# RecursiveCharacterTextSplitter tries progressively smaller separators
# (paragraphs → sentences → words) to keep splits semantically coherent.
# from_tiktoken_encoder uses the GPT-4 tokenizer to count tokens accurately,
# ensuring chunks fit within the embedding model's context window.
#
# chunk_size=300  : maximum tokens per chunk
# chunk_overlap=50: overlap between consecutive chunks to avoid cutting context
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)
splits = text_splitter.split_documents(car_docs)

## 4. Vector Store & Retriever

Each chunk is embedded and stored in a Chroma vector store. The retriever will later perform approximate nearest-neighbour (ANN) search against this store.

In [ ]:
# Embed all document chunks and index them in ChromaDB (in-memory by default).
# Chroma.from_documents calls embd.embed_documents() on every chunk in `splits`
# and stores the resulting vectors for similarity search.
vectorstore = Chroma.from_documents(documents=splits, embedding=embd)

# Wrap the vector store as a LangChain retriever.
# By default, as_retriever() uses cosine similarity and returns the top-4 results.
retriever = vectorstore.as_retriever()

## 5. Multi-Query Generation

A single user question is expanded into **4 diverse search queries** by the LLM. This combats the vocabulary mismatch problem: different phrasings of the same question may retrieve different — and complementary — chunks.

In [ ]:
# Prompt template that instructs the LLM to generate 4 retrieval-optimised queries
template = """
You are an expert retrieval query generator.

Given a user question, generate 4 diverse search queries optimized for semantic and keyword-based retrieval.

Guidelines:
- Preserve the core intent and important entities.
- Use different phrasings and levels of specificity.
- Include:
  1. one precise technical query
  2. one broader conceptual query
  3. one short keyword-style query
  4. one query with likely synonyms or related terminology
- Avoid unnecessary words, explanations, or numbering.
- Maximize retrieval recall while keeping queries concise.

Question: {question}

(Output) 4 Queries:
"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

# Chain: prompt → LLM → parse string output → split on newlines into a list of queries
generate_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: x.split('\n'))  # Each line becomes one query string
)

## 6. Reciprocal Rank Fusion

**Reciprocal Rank Fusion (RRF)** merges the ranked lists returned by each of the 4 queries into a single, de-duplicated ranking.

For each document $d$ appearing in result list $r$ at rank position $i$, the fusion score is:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + i}$$

where $k = 60$ is a smoothing constant that reduces the dominance of top-ranked documents. A higher score means the document was consistently ranked highly across multiple queries.

In [ ]:
def reciprocal_rank_fusion(results: list[list], k: int = 60) -> list[tuple]:
    """
    Merge and re-rank multiple ranked document lists using Reciprocal Rank Fusion.

    For each document appearing in one or more result lists, a cumulative fusion
    score is computed as the sum of 1 / (rank + k) across all lists. Documents
    are then sorted by descending fusion score, so documents that appeared
    near the top of multiple lists bubble to the front.

    Parameters
    ----------
    results : list[list]
        A list of ranked document lists, one per query. Each inner list
        contains LangChain Document objects ordered by relevance.
    k : int, optional
        Smoothing constant (default 60). Higher values reduce the weight of
        top-ranked documents relative to lower-ranked ones.

    Returns
    -------
    list[tuple]
        A de-duplicated list of (Document, fusion_score) tuples sorted by
        descending fusion score.
    """
    fused_ranks = {}  # Maps serialized doc string → cumulative RRF score

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_ranks:
                fused_ranks[doc_str] = 0
            fused_ranks[doc_str] += 1 / (rank + k)

    # Deserialize and sort by descending score
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_ranks.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results


# ── Test: retrieve and fuse results for a sample question ──
question = 'The Gasoline Particular Filter Full warning has appeared. What does this mean and what should I do about it?'

# Full retrieval chain:
#   generate_queries : question → [q1, q2, q3, q4]
#   retriever.map()  : [q1, ..] → [[docs1], [docs2], [docs3], [docs4]]
#   reciprocal_rank_fusion : [[docs1], ..] → [(doc, score), ...]
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion

docs = retrieval_chain_rag_fusion.invoke({'question': question})
print(f'Number of unique documents retrieved after fusion: {len(docs)}')

## 7. Final RAG Chain & Inference

The fused documents are injected as `{context}` into a final answer prompt. The LLM generates a grounded response using only the retrieved context — preventing hallucination on topics outside the manual.

In [ ]:
# Prompt that grounds the LLM's answer strictly in the retrieved context.
# No instructions about tone or language are given here —
# the user can specify preferences (e.g. "réponse en français") in their question.
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# ── Final RAG chain ──
# Input  : {"question": str}
# Output : str (LLM answer grounded in retrieved context)
#
# The chain works as follows:
#   1. retrieval_chain_rag_fusion runs on "question" → fused (doc, score) list as context
#   2. itemgetter("question") extracts the raw question string for the prompt
#   3. prompt formats both into a single LLM-ready message
#   4. llm generates the answer
#   5. StrOutputParser extracts the plain text
final_rag_chain = (
    {'context': retrieval_chain_rag_fusion, 'question': itemgetter('question')}
    | prompt
    | llm
    | StrOutputParser()
)

# ── Example questions you may ask ──
"""    
    "What does the engine oil pressure warning light mean?",
    "How often should I replace the air filter in my car?",
    "What should I do if my car displays a battery warning?",
    "Explain the meaning of the ABS warning light.",
    "Which regular maintenance to do on the car ?",
    "How do I reset the tire pressure monitoring system?",
    "What are the consequences of ignoring the coolant temperature warning?",
    "When should I change my brake pads?",
    "What does the check engine light indicate?",
    "How should you react when the check engine light comes on?"
"""
question = 'What are the warnings to care about?'
answer = final_rag_chain.invoke({'question': question})
print(answer)